# Fase 4.5 — *Backtest* temporal: entrenar con < 2025 y evaluar en 2025+

**Tesis v2 · Eider Johan Díaz Berrío · MCDA EAFIT**

La validación cruzada de los notebooks 09–12 mezcla épocas: un pliegue de prueba puede tener créditos de 2023 calificados por un modelo entrenado con 2025. En la operación ocurre lo contrario, porque el modelo se entrena con el pasado y califica el futuro. Este notebook simula ese uso.

Para cada modelo de la batería:

1. lo entreno **una sola vez** con las cohortes < 2025, eligiendo hiperparámetros con el pliegue interno agrupado **dentro** de < 2025;
2. califico las cohortes 2025+;
3. mido el desempeño con IC por *bootstrap* de clientes (2.000 réplicas) dentro de 2025+.

**Advertencia preespecificada (enmienda v1.1, sección 6).** El conjunto 2025+ **ya fue consultado en la v1** de la tesis, así que no es una prueba ciega. Lo uso como evidencia secundaria de estabilidad temporal. **No interviene en la selección de modelo**, que quedó cerrada en el notebook 10 (M2).

Cuidado adicional contra fuga temporal: los modelos que usan datos auxiliares (M6, supervivencia; M7, objetivos 60+/12m y 30+/6m) también se restringen a cohortes < 2025.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from tesis_v2 import config, contexto, estadistica as est, figuras, io_datos, metricas as mt, modelos

warnings.filterwarnings("ignore")
pd.set_option("display.width", 190, "display.max_columns", 30, "display.max_colwidth", 60)
figuras.estilo()

D, ctx, ds = contexto.cargar()
B = 2000
es_tr = D["anio_cohorte"] < config.ANIO_BACKTEST
TR, TE = D[es_tr].reset_index(drop=True), D[~es_tr].reset_index(drop=True)


def antes_de_corte(df):
    return df[df["anio_cohorte"] < config.ANIO_BACKTEST].reset_index(drop=True)


ctx_bt = modelos.Contexto(grupos_primarios=ctx.grupos_primarios, beta_hc=ctx.beta_hc, X_hc=ctx.X_hc, y_hc=ctx.y_hc,
                          supervivencia=antes_de_corte(ctx.supervivencia),
                          auxiliares={o: antes_de_corte(a) for o, a in ctx.auxiliares.items()})

comunes = set(TR["grupo_cliente"]) & set(TE["grupo_cliente"])
desc = pd.DataFrame([
    {"conjunto": "Entrenamiento (< 2025)", "creditos": len(TR), "eventos": int(TR["y"].sum()),
     "clientes": TR["grupo_cliente"].nunique()},
    {"conjunto": "Prueba temporal (2025+)", "creditos": len(TE), "eventos": int(TE["y"].sum()),
     "clientes": TE["grupo_cliente"].nunique()}])
desc["tasa_%"] = (100 * desc["eventos"] / desc["creditos"]).round(2)
desc["ic95_%"] = [f"{100 * lo:.1f}–{100 * hi:.1f}" for lo, hi in (est.wilson(e, n) for e, n in zip(desc["eventos"], desc["creditos"]))]
io_datos.guardar_resultado(desc, "fase4", "backtest_conjuntos")
print(f"Créditos 2025+ de clientes que ya estaban en < 2025: {TE['grupo_cliente'].isin(comunes).sum()} "
      f"de {len(TE)} ({100 * TE['grupo_cliente'].isin(comunes).mean():.0f} %)")
print("Eventos 2025+ por cohorte:", TE.groupby("anio_cohorte")["y"].agg(["size", "sum"]).to_dict("index"))
desc

Créditos 2025+ de clientes que ya estaban en < 2025: 134 de 237 (57 %)
Eventos 2025+ por cohorte: {np.int64(2025): {'size': 205, 'sum': 5}, np.int64(2026): {'size': 32, 'sum': 0}}


,conjunto,creditos,eventos,clientes,tasa_%,ic95_%
0,Entrenamiento (< 2025),496,18,272,3.63,2.3–5.7
1,Prueba temporal (2025+),237,5,185,2.11,0.9–4.8


Con **5 eventos en 2025+**, el *backtest* tiene muy poco poder. Un AUC puntual puede caer en cualquier parte entre 0,3 y 0,9 solo por azar: con 5 eventos, cada uno mueve el AUC unos 0,2 puntos según dónde quede en el ordenamiento. Por eso me fijo sobre todo en dos cosas:

- **la calibración en grande**: si la tasa predicha coincide con la observada, que es la pregunta operativa del despliegue;
- **el sentido** de las diferencias frente a la validación cruzada, más que en su magnitud.

La tasa observada baja de 3,6 % en < 2025 a 2,1 % en 2025+. Un modelo entrenado con el pasado **debería sobrestimar** el riesgo de 2025+ si no capta lo que cambió.

## 1. Ajuste único con < 2025 y calificación de 2025+

In [2]:
bat = modelos.bateria()
P, elegidos, nombres = {}, {}, {}
for k, crear in bat.items():
    t = time.time()
    m = crear().ajustar(TR, ctx_bt)
    P[k] = m.predecir(TE)[None, :]
    elegidos[k], nombres[k] = m.elegido, m.nombre
    print(f"{k:6s} {time.time() - t:5.1f} s  elegido={m.elegido}")

y_te, g_te = TE["y"].to_numpy(), TE["grupo_cliente"].to_numpy()
boot = mt.bootstrap(P, y_te, g_te, B=B)
tab = mt.tabla_resultados(P, y_te, boot, nombres).drop(columns="auc_de_repeticiones")
tab["elegido_interno"] = [str(elegidos[k]) for k in tab["modelo"]]
io_datos.guardar_resultado(tab.round(5), "fase4", "backtest_resultados")
tab[["modelo", "log_loss", "log_loss_ic", "brier_escalado", "auc", "auc_ic", "intercepto_cal", "pendiente_cal",
     "pd_media", "tasa_observada"]].round(4)

M0a      0.0 s  elegido=None
M0b      0.0 s  elegido=None


M1       0.3 s  elegido=4


M2       0.6 s  elegido=(np.float64(0.1), 0.5)
M2F      0.0 s  elegido=None
M3       0.1 s  elegido=0.25


M4       0.3 s  elegido=None


M5       1.2 s  elegido=(1, 50, 0.1)
M6       0.1 s  elegido=0.01


M7_60    0.1 s  elegido=(np.float64(0.1), 0.0)
M7_30    0.1 s  elegido=(np.float64(0.001), 0.0)


,modelo,log_loss,log_loss_ic,brier_escalado,auc,auc_ic,intercepto_cal,pendiente_cal,pd_media,tasa_observada
0,M0a,0.1066,0.052–0.173,-0.0126,0.5000,0.500–0.500,-0.5844,NaN,0.0372,0.0211
1,M0b,0.1120,0.053–0.181,-0.0291,0.3711,0.078–0.582,-0.6163,-1.7472,0.0384,0.0211
2,M1,0.1228,0.045–0.219,-0.0353,0.5142,0.059–0.882,-0.3824,-0.2201,0.0304,0.0211
3,M2,0.1076,0.046–0.183,-0.0185,0.4129,0.046–0.861,-0.3503,-0.4570,0.0296,0.0211
4,M2F,0.1303,0.059–0.220,-0.2168,0.4129,0.062–0.819,-0.7324,-0.3574,0.0403,0.0211
5,M3,0.1083,0.043–0.191,-0.0085,0.5103,0.144–0.905,-0.2516,0.0354,0.0269,0.0211
6,M4,0.1155,0.042–0.209,-0.0301,0.5034,0.135–0.890,-0.1366,-0.0428,0.0240,0.0211
7,M5,0.1137,0.048–0.196,-0.0295,0.4466,0.091–0.859,-0.3960,-0.3318,0.0308,0.0211
8,M6,0.1155,0.067–0.174,-0.0535,0.5500,0.226–0.858,-0.9569,0.3821,0.0530,0.0211
9,M7_60,0.1173,0.040–0.213,-0.0494,0.4897,0.144–0.877,-0.0754,-0.1297,0.0226,0.0211


**Lectura.** El resultado es negativo y lo reporto tal cual.

- **Ningún modelo discrimina en 2025+.** El AUC va de 0,37 (M0b) a 0,59 (M7_30), y el del modelo elegido M2 es **0,41**. Todos los IC son enormes (por ejemplo, 0,05–0,86 para M2), porque con 5 eventos el AUC es casi una lotería. Aun así, ningún punto se acerca al 0,63–0,65 de la validación cruzada.
- **Ningún modelo mejora a la tasa base en log loss.** M2 y M3 empatan con ella (0,108 frente a 0,107). Los modelos más flexibles (M1, M2F) empeoran.
- **Todos sobrestiman el riesgo de 2025+** (intercepto de calibración negativo), como se anticipaba: la tasa bajó de 3,6 % a 2,1 %. Los que más se acercan a la tasa observada son los más contraídos o los que toman información de otras fuentes (M7_60, M4, M3). M6 es el que más sobrestima (5,3 %).
- **Las pendientes de calibración son cercanas a 0 o negativas**: el ordenamiento aprendido en < 2025 no se traslada a 2025+.

## 2. Comparaciones pareadas en 2025+

Las mismas que en el notebook 10: cada modelo frente a la tasa base en log loss, y M3 frente a M2. Las réplicas *bootstrap* son las mismas para todos, así que las diferencias son pareadas.

In [3]:
comp = pd.DataFrame([mt.comparacion_pareada(boot, tab, k, "M0a", "log_loss") for k in P if k != "M0a"] +
                    [mt.comparacion_pareada(boot, tab, "M3", "M2", "log_loss"),
                     mt.comparacion_pareada(boot, tab, "M2", "M0b", "log_loss")])
for c in ["delta", "ic_inf", "ic_sup"]:
    comp[c] = 1000 * comp[c]
comp = comp.rename(columns={"delta": "delta_x1000", "ic_inf": "ic_inf_x1000", "ic_sup": "ic_sup_x1000"})
io_datos.guardar_resultado(comp.round(3), "fase4", "backtest_comparaciones")
comp.round(2)

,modelo,frente_a,metrica,delta_x1000,ic_inf_x1000,ic_sup_x1000,p_bootstrap
0,M0b,M0a,log_loss,5.45,-0.94,14.83,0.14
1,M1,M0a,log_loss,16.28,-12.91,59.70,0.41
2,M2,M0a,log_loss,1.09,-9.62,14.41,0.96
3,M2F,M0a,log_loss,23.77,-4.70,59.43,0.12
4,M3,M0a,log_loss,1.73,-16.80,26.65,0.98
5,M4,M0a,log_loss,8.95,-18.54,46.18,0.68
6,M5,M0a,log_loss,7.16,-12.25,31.45,0.59
7,M6,M0a,log_loss,8.90,0.24,15.83,0.05
8,M7_60,M0a,log_loss,10.77,-15.80,47.29,0.57
9,M7_30,M0a,log_loss,-2.18,-14.48,17.94,0.69


## 3. Validación cruzada frente a *backtest*

Pongo lado a lado las métricas de la validación cruzada (notebook 10, todas las cohortes) y las del *backtest*. Si el modelo fuera estable en el tiempo, el *backtest* debería caer dentro del IC de la validación cruzada.

In [4]:
cv = pd.read_csv(config.RESULTADOS / "fase4" / "tabla_unica_resultados.csv").set_index("modelo")
T = tab.set_index("modelo")
lado = pd.DataFrame({
    "auc_cv": cv["auc"], "auc_cv_ic": cv["auc_ic"], "auc_backtest": T["auc"], "auc_backtest_ic": T["auc_ic"],
    "log_loss_cv": cv["log_loss"], "log_loss_backtest": T["log_loss"],
    "pendiente_cv": cv["pendiente_cal"], "pendiente_backtest": T["pendiente_cal"],
    "pd_media_backtest": T["pd_media"], "tasa_obs_backtest": T["tasa_observada"]}).loc[list(P)]
io_datos.guardar_resultado(lado.reset_index().round(4), "fase4", "backtest_vs_validacion_cruzada")
lado.round(3)

,auc_cv,auc_cv_ic,auc_backtest,auc_backtest_ic,log_loss_cv,log_loss_backtest,pendiente_cv,pendiente_backtest,pd_media_backtest,tasa_obs_backtest
modelo,,,,,,,,,,
M0a,0.474,0.451–0.496,0.500,0.500–0.500,0.140,0.107,-4.239,NaN,0.037,0.021
M0b,0.391,0.319–0.448,0.371,0.078–0.582,0.142,0.112,-2.550,-1.747,0.038,0.021
M1,0.547,0.434–0.657,0.514,0.059–0.882,0.151,0.123,0.077,-0.220,0.030,0.021
M2,0.629,0.502–0.747,0.413,0.046–0.861,0.139,0.108,0.618,-0.457,0.030,0.021
M2F,0.600,0.482–0.719,0.413,0.062–0.819,0.148,0.130,0.222,-0.357,0.040,0.021
M3,0.653,0.525–0.781,0.510,0.144–0.905,0.137,0.108,0.772,0.035,0.027,0.021
M4,0.644,0.518–0.776,0.503,0.135–0.890,0.138,0.116,0.575,-0.043,0.024,0.021
M5,0.633,0.515–0.754,0.447,0.091–0.859,0.137,0.114,0.724,-0.332,0.031,0.021
M6,0.602,0.481–0.725,0.550,0.226–0.858,0.143,0.115,1.169,0.382,0.053,0.021


**Lectura.** En la validación cruzada, todos los modelos principales tienen AUC entre 0,60 y 0,65. En el *backtest* caen a 0,41–0,55. El *backtest* queda dentro del IC de la validación cruzada solo porque su propio IC es enorme.

El log loss del *backtest* es menor que el de la validación cruzada, pero eso no es una mejora: 2025+ tiene menos eventos (2,1 % frente a 3,1 %), y el log loss baja con la tasa. Por eso la comparación justa es siempre **contra la tasa base dentro del mismo conjunto**, como en la sección 2.

## 3b. ¿Por qué no se sostiene? Relaciones univariadas por cohorte

Para entender el resultado, calculo el AUC univariado **sin orientar** de cada predictor dentro de cada cohorte. Un valor mayor que 0,5 indica que valores altos de la variable acompañan más incumplimientos. Si una variable tuviera una relación estable con el riesgo, su AUC quedaría del mismo lado de 0,5 en todas las cohortes.

In [5]:
from sklearn.metrics import roc_auc_score

from tesis_v2 import diseno

X = diseno.construir_X(D)
cohorte = np.where(D["anio_cohorte"] >= config.ANIO_BACKTEST, "2025+", D["anio_cohorte"].astype(str))
filas = []
for v in diseno.NUCLEO:
    fila = {"variable": v}
    for c in ["2022", "2023", "2024", "2025+"]:
        m = (cohorte == c) & X[v].notna().to_numpy()
        fila[c] = roc_auc_score(D["y"][m], X[v][m])
    filas.append(fila)
uni = pd.DataFrame(filas).set_index("variable")
ev = pd.Series({c: int(D["y"][cohorte == c].sum()) for c in uni.columns})
uni["cambia_de_lado"] = ((uni[["2022", "2023", "2024"]] - 0.5).mean(axis=1) * (uni["2025+"] - 0.5) < 0)
io_datos.guardar_resultado(uni.reset_index().round(3), "fase4", "backtest_auc_univariado_por_cohorte")
coef_tr = modelos.M2Penalizada().ajustar(TR, ctx_bt).coeficientes(TR)
coef_todo = modelos.M2Penalizada().ajustar(D, ctx).coeficientes(D)
print("Eventos por cohorte:", ev.to_dict())
print("AUC de M2 en su propio entrenamiento (< 2025):",
      round(roc_auc_score(TR["y"], modelos.M2Penalizada().ajustar(TR, ctx_bt).predecir(TR)), 3))
display(pd.DataFrame({"coef_M2_<2025": coef_tr, "coef_M2_todo": coef_todo}).round(3))
uni.round(2)

Eventos por cohorte: {'2022': 6, '2023': 5, '2024': 7, '2025+': 5}


AUC de M2 en su propio entrenamiento (< 2025): 0.736


,coef_M2_<2025,coef_M2_todo
log_monto,0.163,0.239
log_plazo,0.000,0.048
edad_t0,-0.018,-0.199
tiene_historia,0.000,0.031
log_n_previos,0.000,-0.016
meses_cliente,0.000,-0.168
log_peor_mora_previa,0.000,-0.031
previo_adverso,0.000,0.045
previo_favorable,0.000,-0.071
n_activos_t0,0.000,-0.134


,2022,2023,2024,2025+,cambia_de_lado
variable,,,,,
log_monto,0.62,0.77,0.64,0.37,True
log_plazo,0.23,0.65,0.64,0.36,True
edad_t0,0.20,0.54,0.42,0.43,False
tiene_historia,0.47,0.60,0.48,0.37,True
log_n_previos,0.47,0.62,0.42,0.38,True
meses_cliente,0.47,0.69,0.35,0.33,True
log_peor_mora_previa,0.49,0.58,0.35,0.46,False
previo_adverso,0.50,0.65,0.52,0.37,True
previo_favorable,0.50,0.56,0.40,0.47,False


In [6]:
fig, ax = plt.subplots(figsize=(6.4, 4.6))
M = uni[["2022", "2023", "2024", "2025+"]].to_numpy(float)
im = ax.imshow(M, cmap="RdBu_r", vmin=0.2, vmax=0.8, aspect="auto")
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=7)
ax.set_xticks(range(4), [f"{c}\n({ev[c]} eventos)" for c in ["2022", "2023", "2024", "2025+"]])
ax.set_yticks(range(len(uni)), uni.index)
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.8, label="AUC univariado (> 0,5: más valor, más riesgo)")
ax.set_title("Relación de cada predictor con el incumplimiento, por cohorte")
fig.tight_layout()
figuras.guardar(fig, "fase4", "M12", "auc_univariado_por_cohorte",
                "Ningún predictor mantiene una relación estable con el incumplimiento entre cohortes: el monto, la señal más regular antes de 2025, se invierte en 2025+.")

**Lectura.** Esta tabla explica el *backtest*.

- **Con su propio entrenamiento, M2 tiene AUC 0,74.** El modelo sí aprende un patrón en < 2025, pero ese patrón no se repite en 2025+.
- **El monto** es la señal más regular antes de 2025 (AUC entre 0,62 y 0,77 en las tres cohortes) y **se invierte en 2025+** (0,37). Es el coeficiente más grande de M2 junto con el empleador, y por eso el modelo ordena al revés.
- **Los años con el empleador** protegen en 2022 y 2024 (0,27 y 0,28), no discriminan en 2023 (0,50) y tampoco en 2025+ (0,49).
- **La antigüedad como cliente y la historia previa** cambian de lado entre 2023 y 2024.

Con 5 a 7 eventos por cohorte, ninguna de estas inversiones es estadísticamente distinguible del azar. Esa es justamente la conclusión: **con este volumen de eventos no hay ninguna relación predictor-riesgo que se reproduzca de una cohorte a otra.**

El AUC de 0,63 de la validación cruzada (que mezcla cohortes) se apoyaba en patrones que, mirados año por año, no son estables. Esto es coherente con lo que mostró el notebook 12: sin la cohorte 2022, la señal desaparece.

## 4. Figuras

In [7]:
orden = [k for k in P if k != "M0a"]
fig, ax = plt.subplots(1, 2, figsize=(10, 4.2), sharey=True)
for i, k in enumerate(orden):
    lo, hi = np.nanpercentile(boot[k]["auc"], [2.5, 97.5])
    ax[0].errorbar(T.loc[k, "auc"], i, xerr=[[T.loc[k, "auc"] - lo], [hi - T.loc[k, "auc"]]], fmt="o",
                   color="#c0392b" if k == "M2" else "#34495e", capsize=3)
    ax[0].plot(cv.loc[k, "auc"], i, marker="|", color="#7f8c8d", markersize=12, mew=2)
    c = comp[(comp["modelo"] == k) & (comp["frente_a"] == "M0a")].iloc[0]
    ax[1].errorbar(c["delta_x1000"], i, xerr=[[c["delta_x1000"] - c["ic_inf_x1000"]], [c["ic_sup_x1000"] - c["delta_x1000"]]],
                   fmt="o", color="#c0392b" if k == "M2" else "#34495e", capsize=3)
ax[0].axvline(0.5, color="k", lw=0.8, ls="--")
ax[1].axvline(0, color="k", lw=0.8, ls="--")
ax[0].set_yticks(range(len(orden)), [nombres[k] for k in orden])
ax[0].set_xlabel("AUC en 2025+ (IC 95 %); | = AUC en validación cruzada")
ax[1].set_xlabel("Δ log loss frente a la tasa base × 1.000 (negativo = mejor)")
ax[0].set_title("Discriminación en el backtest")
ax[1].set_title("Mejora probabilística en el backtest")
fig.suptitle(f"Backtest temporal: entrenamiento < 2025 ({len(TR)} créditos, {int(TR['y'].sum())} eventos) → "
             f"prueba 2025+ ({len(TE)} créditos, {int(TE['y'].sum())} eventos)", fontsize=10)
fig.tight_layout()
MENSAJE_M10 = "En 2025+ ningún modelo discrimina (AUC 0,37–0,59, con IC que cubren casi todo el rango) ni mejora a la tasa base: el AUC de 0,63 de la validación cruzada no se sostiene en el tiempo."
figuras.guardar(fig, "fase4", "M10", "backtest_temporal", MENSAJE_M10)

In [8]:
def captura(y, p):
    o = np.argsort(-p, kind="stable")
    return np.r_[0, np.arange(1, len(y) + 1) / len(y)], np.r_[0, np.cumsum(y[o]) / y.sum()]


fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for k, col in [("M2", "#c0392b"), ("M3", "#2980b9"), ("M0b", "#7f8c8d")]:
    x, yy = captura(y_te, P[k][0])
    ax[0].step(x, yy, where="post", color=col, label=nombres[k])
ax[0].plot([0, 1], [0, 1], "k--", lw=0.8, label="Azar")
ax[0].set_xlabel("Fracción de créditos 2025+ revisados (de mayor a menor PD)")
ax[0].set_ylabel("Fracción de los 5 incumplimientos capturados")
ax[0].legend(fontsize=7)
ax[0].set_title("Curva de captura en 2025+")
x = np.arange(len(orden))
ax[1].bar(x, [100 * T.loc[k, "pd_media"] for k in orden], color="#95a5a6", label="PD media predicha")
lo, hi = est.wilson(int(y_te.sum()), len(y_te))
ax[1].axhspan(100 * lo, 100 * hi, color="#c0392b", alpha=0.12, label="Tasa observada 2025+ (IC 95 %)")
ax[1].axhline(100 * y_te.mean(), color="#c0392b", lw=1.2)
ax[1].axhline(100 * TR["y"].mean(), color="k", lw=0.8, ls=":", label="Tasa del entrenamiento < 2025")
ax[1].set_xticks(x, orden, rotation=45)
ax[1].set_ylabel("%")
ax[1].legend(fontsize=7)
ax[1].set_title("Calibración en grande")
fig.tight_layout()
MENSAJE_M11 = "En 2025+, M2, M3 y la regla capturan 2 de los 5 incumplimientos en el 15 % de mayor PD, pero los otros 3 quedan al fondo del ordenamiento; todos los modelos sobrestiman la PD (2,3–5,3 % frente a 2,1 % observada), aunque solo M6 sale del IC de la tasa observada."
figuras.guardar(fig, "fase4", "M11", "backtest_captura_calibracion", MENSAJE_M11)

## 5. Origen móvil anual (descriptivo, opcional en el protocolo)

Para ver si el resultado de 2025+ es una rareza de ese año, repito el ejercicio con ventana creciente: entreno con todo lo anterior a un año y evalúo ese año. Solo uso M0a, M2 y M3, y no calculo IC: con 5 a 7 eventos por año, este análisis es **descriptivo**.

In [9]:
filas = []
for anio in [2023, 2024, 2025]:
    tr, te = D[D["anio_cohorte"] < anio].reset_index(drop=True), D[D["anio_cohorte"] >= anio].reset_index(drop=True)
    if anio < 2025:
        te = te[te["anio_cohorte"] == anio].reset_index(drop=True)
    cx = modelos.Contexto(grupos_primarios=ctx.grupos_primarios, beta_hc=ctx.beta_hc, X_hc=ctx.X_hc, y_hc=ctx.y_hc,
                          supervivencia=ctx.supervivencia[ctx.supervivencia["anio_cohorte"] < anio],
                          auxiliares={o: a[a["anio_cohorte"] < anio] for o, a in ctx.auxiliares.items()})
    for k in ["M0a", "M2", "M3"]:
        p = bat[k]().ajustar(tr, cx).predecir(te)
        r = mt.puntuales(p[None, :], te["y"].to_numpy())
        filas.append({"anio_prueba": f"{anio}" if anio < 2025 else "2025+", "modelo": k, "creditos_entrenamiento": len(tr),
                      "eventos_entrenamiento": int(tr["y"].sum()), "creditos_prueba": len(te),
                      "eventos_prueba": int(te["y"].sum()), "auc": r["auc"], "log_loss": r["log_loss"],
                      "pd_media": r["pd_media"], "tasa_observada": r["tasa_observada"]})
movil = pd.DataFrame(filas)
io_datos.guardar_resultado(movil.round(4), "fase4", "backtest_origen_movil")
movil.round(4)

,anio_prueba,modelo,creditos_entrenamiento,eventos_entrenamiento,creditos_prueba,eventos_prueba,auc,log_loss,pd_media,tasa_observada
0,2023,M0a,49,6,197,5,0.5000,0.1875,0.1300,0.0254
1,2023,M2,49,6,197,5,0.4635,0.2196,0.1060,0.0254
2,2023,M3,49,6,197,5,0.5135,0.2110,0.1165,0.0254
3,2024,M0a,246,11,250,7,0.5000,0.1322,0.0466,0.0280
4,2024,M2,246,11,250,7,0.4321,0.2204,0.0892,0.0280
5,2024,M3,246,11,250,7,0.5479,0.1398,0.0486,0.0280
6,2025+,M0a,496,18,237,5,0.5000,0.1066,0.0372,0.0211
7,2025+,M2,496,18,237,5,0.4129,0.1076,0.0296,0.0211
8,2025+,M3,496,18,237,5,0.5103,0.1083,0.0269,0.0211


**Lectura.** El origen móvil confirma que 2025+ no es una rareza:

- **2024**, entrenando con 2022–2023: M2 tiene AUC 0,43 y log loss mucho peor que la tasa base (0,220 frente a 0,132). M3, gracias al *prior*, queda casi en la tasa base (0,140) con AUC 0,55.
- **2023**, entrenando solo con 2022 (6 eventos): la tasa base del entrenamiento (13 %) sobrestima por mucho la de 2023 (2,5 %). Ningún modelo discrimina.

En todas las ventanas, **M3 es el que menos se degrada**. Es lo esperable de un *prior* informativo: limita cuánto puede equivocarse el modelo cuando los datos propios son pocos e inestables. Aun así, nunca supera de forma clara a la tasa base.

## 6. Conclusión del *backtest* y consecuencias para la tesis

1. **La selección de M2 no cambia.** El protocolo prohíbe usar 2025+ para elegir modelo, y M2 tampoco es peor que las alternativas en el *backtest*: todas fallan por igual.
2. **La conclusión principal se refuerza.** Con 23 eventos, los datos de FINTECH no permiten demostrar que un modelo estadístico supere a la tasa base. El *backtest* agrega algo más: la discriminación de la validación cruzada **no se reproduce entre épocas**.
3. **Para el despliegue (Fase 5 y Fase 6) se siguen tres cosas:**
   - el modelo debe usarse como **apoyo de ordenamiento con bandas amplias** y no como decisor automático;
   - la **recalibración** del intercepto a la tasa reciente es necesaria desde el primer día, porque todos los modelos sobrestiman 2025+;
   - el **aprendizaje incremental** no es un añadido: es la única forma de que el modelo siga a una relación predictor-riesgo que cambia de cohorte en cohorte. Debe ir acompañado de un **monitoreo de estabilidad de coeficientes** y de reglas de actualización conservadoras, como un *prior* centrado en el modelo anterior, en la línea de M3.
4. **Limitación explícita.** El conjunto 2025+ ya fue consultado en la v1 y tiene solo 5 eventos. Este resultado no es una prueba definitiva de que el modelo no sirva, sino una evidencia de que **no se puede afirmar que sirva fuera de la muestra temporal**.

## 7. Parámetros del modelo final (para la ficha del modelo)

El modelo que se despliega es M2 ajustado con **todos** los 733 créditos. La penalización se elige con el mismo pliegue interno agrupado. Exporto los parámetros que hacen falta para reproducir la predicción fuera de Python: la mediana de imputación, la media y la desviación estándar de cada predictor, el coeficiente estandarizado, el coeficiente por unidad natural y el intercepto. Son agregados de la cartera, sin filas de clientes.

In [10]:
final = modelos.M2Penalizada().ajustar(D, ctx)
imp, esc, lr = final.m.pipe[0], final.m.pipe[1], final.m.pipe[-1]
par = pd.DataFrame({"variable": diseno.NUCLEO, "mediana_imputacion": imp.statistics_, "media": esc.mean_, "de": esc.scale_,
                    "coef_estandarizado": lr.coef_[0]})
par["coef_por_unidad"] = par["coef_estandarizado"] / par["de"]
intercepto_natural = float(lr.intercept_[0] - (par["coef_estandarizado"] * par["media"] / par["de"]).sum())
par.loc[len(par)] = {"variable": "(intercepto)", "coef_estandarizado": float(lr.intercept_[0]), "coef_por_unidad": intercepto_natural}
io_datos.guardar_resultado(par.round(6), "fase4", "parametros_modelo_final_M2")
p_final = final.predecir(D)
C_f, l1_f = final.elegido
print(f"Configuración elegida con los 733 créditos: {'L2' if l1_f == 0 else 'L1' if l1_f == 1 else f'EN {l1_f}'}, C = {C_f:.3g}")
print(f"PD ajustada en la cartera: mínimo {100 * p_final.min():.2f} %, mediana {100 * np.median(p_final):.2f} %, máximo {100 * p_final.max():.2f} %")
Xf = diseno.construir_X(D).fillna(dict(zip(diseno.NUCLEO, imp.statistics_)))
manual = 1 / (1 + np.exp(-(intercepto_natural + Xf[diseno.NUCLEO].to_numpy() @ par["coef_por_unidad"].iloc[:-1].to_numpy())))
assert np.allclose(manual, p_final), "La fórmula en unidades naturales no reproduce la predicción"
par.round(4)

Configuración elegida con los 733 créditos: L2, C = 0.1
PD ajustada en la cartera: mínimo 0.34 %, mediana 2.62 %, máximo 13.17 %


,variable,mediana_imputacion,media,de,coef_estandarizado,coef_por_unidad
0,log_monto,14.9141,14.9534,1.0811,0.2386,0.2207
1,log_plazo,2.6125,2.6522,0.7410,0.0475,0.0641
2,edad_t0,37.3210,39.2979,10.9716,-0.1986,-0.0181
3,tiene_historia,1.0000,0.5307,0.4991,0.0310,0.0622
4,log_n_previos,0.6931,0.5408,0.5747,-0.0163,-0.0284
5,meses_cliente,3.1537,7.8636,10.1765,-0.1677,-0.0165
6,log_peor_mora_previa,0.0000,0.9944,1.3896,-0.0313,-0.0225
7,previo_adverso,0.0000,0.1951,0.3963,0.0452,0.1141
8,previo_favorable,0.0000,0.1705,0.3761,-0.0706,-0.1876
9,n_activos_t0,0.0000,0.3793,0.5178,-0.1339,-0.2585


In [11]:
_ = io_datos.escribir_registro("13_backtest_temporal", {
    "entrenamiento": {"creditos": len(TR), "eventos": int(TR["y"].sum())},
    "prueba": {"creditos": len(TE), "eventos": int(TE["y"].sum())},
    "consultado_en_v1": True, "interviene_en_seleccion": False, "bootstrap": B,
    "auc_M2": float(T.loc["M2", "auc"]), "auc_M3": float(T.loc["M3", "auc"])})
figuras.escribir_catalogo("fase4", "Figuras de la Fase 4")
with pd.ExcelWriter(config.RESULTADOS / "fase4" / "p6_v2_modelos.xlsx") as xw:
    for f in sorted((config.RESULTADOS / "fase4").glob("*.csv")):
        pd.read_csv(f).to_excel(xw, sheet_name=f.stem[:31], index=False)